In [81]:
import pandas as pd
import sklearn
import os
import numpy as np
from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, r2_score, precision_score, recall_score,
    confusion_matrix, classification_report,
)
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import xgboost as xgb

# Лимит RAM для CatBoost (подстрой под объём ОЗУ; при OOM уменьши до '4gb' или '3gb')
USED_RAM_LIMIT = '8gb'
# Количество потоков: -1 = все ядра; при OOM уменьши до 8 или 4
THREAD_COUNT = -1


In [82]:
df = pd.read_csv("dataset_combined.csv")

In [83]:
len(df)

124425

In [84]:
df.head(5)

,class,region,center_1500,wave_2002.46,wave_2001.50,wave_2000.54,wave_1999.58,wave_1998.62,wave_1997.66,wave_1996.70,...,wave_933.84,wave_932.67,wave_931.50,wave_930.32,wave_929.15,wave_927.98,wave_926.80,source_file,X,Y
0,endo,cortex,True,10178.839844,10165.766602,9905.018555,9887.439453,10146.862305,10076.990234,10103.795898,...,8520.928711,8504.200195,8459.643555,8528.562500,8428.372070,8516.528320,8544.746094,cortex_endo_1group_633nm_center1500_obj100_pow...,8315.753404,13149.765
1,endo,cortex,True,8581.288086,8638.810547,8600.557617,8402.886719,8733.147461,8773.278320,8492.099609,...,6477.447754,6480.410156,6346.338379,6430.672852,6350.172852,6303.942871,6360.422852,cortex_endo_1group_633nm_center1500_obj100_pow...,8317.753404,13149.765
2,endo,cortex,True,9784.027344,9825.862305,9821.366211,9712.325195,9629.458008,9617.164063,9691.076172,...,7805.496094,7752.506348,7866.547852,7751.487305,7670.717773,7921.655273,7821.629883,cortex_endo_1group_633nm_center1500_obj100_pow...,8319.753404,13149.765
3,endo,cortex,True,10320.031250,10461.222656,10365.109375,10535.625000,10562.352539,10356.543945,10393.744141,...,8771.543945,8848.993164,8930.694336,8686.974609,8734.430664,8784.007813,8869.933594,cortex_endo_1group_633nm_center1500_obj100_pow...,8321.753404,13149.765
4,endo,cortex,True,10223.289063,10265.124023,10192.575195,10159.259766,10431.695313,10241.587891,10383.295898,...,8349.567383,8525.615234,8418.961914,8415.105469,8469.037109,8640.638672,8634.600586,cortex_endo_1group_633nm_center1500_obj100_pow...,8323.753404,13149.765


## Предобработка спектров (по рекомендациям PDF)

В методичке указано: *«обратить внимание на подавление шумов, коррекцию фонового сигнала и нормировку спектров»*. Выполняем:
1. **Коррекция фона** — вычитание базовой линии (минимум по спектру или полином).
2. **Сглаживание** (подавление шумов) — фильтр Савицкого–Голея (если доступен scipy).
3. **Нормировка** — по L2-норме (каждый спектр как вектор единичной длины) или по максимуму.

In [85]:
# Столбцы с интенсивностями (волновые числа)
wave_cols = [c for c in df.columns if c.startswith("wave_")]
X_spectra = df[wave_cols].values.astype(np.float64)

# 1. Коррекция фонового сигнала: вычитаем минимум по каждому спектру (простая базовая линия)
baseline = X_spectra.min(axis=1, keepdims=True)
X_spectra = X_spectra - baseline
# Не допускаем отрицательных интенсивностей
X_spectra = np.maximum(X_spectra, 0.0)

# 2. Подавление шумов: сглаживание Савицкого–Голея (опционально)
try:
    from scipy.signal import savgol_filter
    # window_length нечётное, polyorder обычно 2–5
    X_spectra = savgol_filter(X_spectra, window_length=11, polyorder=3, axis=1)
    print("Применён фильтр Савицкого–Голея (window=11, poly=3)")
except ImportError:
    print("scipy не установлен — шаг сглаживания пропущен")

# 3. Нормировка спектров: L2-норма (каждый спектр — вектор единичной длины)
norms = np.linalg.norm(X_spectra, axis=1, keepdims=True)
norms[norms == 0] = 1.0  # чтобы не делить на 0
X_spectra = X_spectra / norms

df[wave_cols] = X_spectra
print("Предобработка применена: коррекция фона + нормировка L2.")

Применён фильтр Савицкого–Голея (window=11, poly=3)
Предобработка применена: коррекция фона + нормировка L2.


## ⚠️ Проверка на data leakage (утечку по группам)

При случайном split по строкам строки из **одного source_file** попадают и в train, и в test. Спектры из одного файла (одного образца) почти идентичны — модель "подглядывает" в тесте. Ниже — диагностика.

In [86]:
# Диагностика: сколько source_file попадают и в train, и в test?
from sklearn.model_selection import train_test_split

x_tmp = df.drop(['class','source_file','X','Y'], axis=1)
y_tmp = df['class']
x_tr, x_te, y_tr, y_te = train_test_split(x_tmp, y_tmp, test_size=0.2, random_state=42, shuffle=True)

train_files = set(df.loc[x_tr.index, 'source_file'])
test_files = set(df.loc[x_te.index, 'source_file'])
overlap = train_files & test_files

print(f"Уникальных source_file: {df['source_file'].nunique()}")
print(f"Файлов в train: {len(train_files)}, в test: {len(test_files)}")
print(f"Файлов в ОБОИХ (утечка!): {len(overlap)} ({100*len(overlap)/len(test_files):.1f}% теста)")

Уникальных source_file: 237
Файлов в train: 237, в test: 237
Файлов в ОБОИХ (утечка!): 237 (100.0% теста)


In [87]:
from sklearn.model_selection import GroupShuffleSplit

x = df.drop(['class','source_file','X','Y'], axis=1)
y = df['class']
groups = df['source_file']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(x, y, groups))

x_train, x_test = x.iloc[train_idx], x.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

In [88]:
# Проверка: после split по группам утечки быть не должно
train_files_new = set(df.loc[train_idx, 'source_file'])
test_files_new = set(df.loc[test_idx, 'source_file'])
print("Файлов в train:", len(train_files_new), "| в test:", len(test_files_new))
print("Пересечение (должно быть 0):", len(train_files_new & test_files_new))

Файлов в train: 189 | в test: 48
Пересечение (должно быть 0): 0


In [89]:
# Закодируем region для XGB, GB, RF (они не поддерживают категориальные столбцы как CatBoost/LGBM)
x_train_enc = pd.get_dummies(x_train, columns=['region'])
x_test_enc = pd.get_dummies(x_test, columns=['region'])
# Выровняем столбцы теста под train (на случай отсутствующих категорий в test)
for c in x_train_enc.columns:
    if c not in x_test_enc.columns:
        x_test_enc[c] = 0
x_test_enc = x_test_enc[x_train_enc.columns]

# Метки для sklearn/XGB/LGBM: exo=0, endo=1, иначе 2
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train_num = le.fit_transform(y_train)
y_test_num = le.transform(y_test)
# Соответствие: le.classes_ -> 0,1,2,...
print("Классы (метки):", dict(zip(le.classes_, range(len(le.classes_)))))

Классы (метки): {'control': 0, 'endo': 1, 'exo': 2}


In [90]:
x_train.head(5)

,region,center_1500,wave_2002.46,wave_2001.50,wave_2000.54,wave_1999.58,wave_1998.62,wave_1997.66,wave_1996.70,wave_1995.74,...,wave_937.35,wave_936.18,wave_935.01,wave_933.84,wave_932.67,wave_931.50,wave_930.32,wave_929.15,wave_927.98,wave_926.80
525,cortex,True,0.012334,0.011624,0.010891,0.010217,0.009680,0.009360,0.010516,0.010032,...,0.019660,0.019335,0.019689,0.019697,0.019596,0.020112,0.020578,0.020847,0.020769,0.020198
526,cortex,True,0.042588,0.041884,0.041455,0.041285,0.041360,0.041664,0.042835,0.043318,...,0.004082,0.004335,0.004640,0.005328,0.006093,0.006868,0.007019,0.006182,0.003996,0.000098
527,cortex,True,0.037502,0.039266,0.039589,0.038845,0.037408,0.035650,0.034506,0.033849,...,0.006253,0.006180,0.005479,0.005067,0.004631,0.004102,0.003601,0.003177,0.002879,0.002756
528,cortex,True,0.042517,0.042326,0.042388,0.042609,0.042892,0.043142,0.042686,0.042355,...,0.002841,0.003168,0.003553,0.004089,0.003960,0.003936,0.003612,0.002894,0.001687,-0.000103
529,cortex,True,0.044735,0.044027,0.043225,0.042405,0.041641,0.041008,0.040395,0.040780,...,0.002022,0.002532,0.002604,0.002797,0.002623,0.002422,0.002196,0.001985,0.001830,0.001773


In [91]:
# --- Стеккинг: OOF-предсказания базовых моделей + мета-модель ---
RANDOM_STATE = 42
N_ESTIMATORS = 300
MAX_DEPTH = 8
N_FOLDS = 5

groups_train = groups.iloc[train_idx].values
gkf = GroupKFold(n_splits=N_FOLDS)
n_classes = len(le.classes_)
oof = np.zeros((len(y_train_num), n_classes * 3))  # LGBM + XGB + CatBoost proba

for fold, (tr_idx, val_idx) in enumerate(gkf.split(x_train_enc, y_train_num, groups_train)):
    x_tr, x_val = x_train_enc.iloc[tr_idx], x_train_enc.iloc[val_idx]
    y_tr, y_val = y_train_num[tr_idx], y_train_num[val_idx]

    m_lgbm = lgb.LGBMClassifier(n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, learning_rate=0.05,
        num_leaves=2**MAX_DEPTH, subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE,
        n_jobs=THREAD_COUNT, verbose=-1)
    m_lgbm.fit(x_tr, y_tr, eval_set=[(x_val, y_val)], callbacks=[lgb.early_stopping(50, verbose=False)])

    m_xgb = xgb.XGBClassifier(n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE,
        n_jobs=THREAD_COUNT if THREAD_COUNT > 0 else None, eval_metric='mlogloss')
    m_xgb.fit(x_tr, y_tr, eval_set=[(x_val, y_val)], verbose=False)

    m_cat = CatBoostClassifier(iterations=N_ESTIMATORS, depth=MAX_DEPTH, learning_rate=0.05,
        thread_count=THREAD_COUNT, used_ram_limit=USED_RAM_LIMIT, max_bin=128,
        bootstrap_type='Bernoulli', subsample=0.8, random_state=RANDOM_STATE, verbose=0)
    m_cat.fit(x_tr, y_tr, eval_set=(x_val, y_val), early_stopping_rounds=50, verbose=0)

    oof[val_idx, 0:n_classes] = m_lgbm.predict_proba(x_val)
    oof[val_idx, n_classes:2*n_classes] = m_xgb.predict_proba(x_val)
    oof[val_idx, 2*n_classes:3*n_classes] = m_cat.predict_proba(x_val)

print("OOF готов. Форма oof:", oof.shape)

OOF готов. Форма oof: (99225, 9)


In [92]:
# Обучение мета-модели на OOF-предсказаниях (без multi_class — совместимость со всеми версиями sklearn)
meta_learner = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
meta_learner.fit(oof, y_train_num)
print("Мета-модель обучена на", oof.shape[0], "объектах,", oof.shape[1], "признаков.")

Мета-модель обучена на 99225 объектах, 9 признаков.


In [93]:
# Стеккинг: предсказания базовых моделей на тесте -> мета-модель
p_lgbm = m_lgbm.predict_proba(x_test_enc)
p_xgb  = m_xgb.predict_proba(x_test_enc)
p_cat  = m_cat.predict_proba(x_test_enc)

# Склеиваем вероятности в один вектор признаков (как при OOF)
test_stack = np.hstack([p_lgbm, p_xgb, p_cat])
y_pred_ensemble = meta_learner.predict(test_stack)

In [94]:
# Оценка каждой модели и ансамбля (расширенные метрики)
def eval_model(name, y_pred):
    acc = accuracy_score(y_test_num, y_pred)
    p_macro = precision_score(y_test_num, y_pred, average='macro', zero_division=0)
    p_micro = precision_score(y_test_num, y_pred, average='micro', zero_division=0)
    p_wtd   = precision_score(y_test_num, y_pred, average='weighted', zero_division=0)
    r_macro = recall_score(y_test_num, y_pred, average='macro', zero_division=0)
    r_micro = recall_score(y_test_num, y_pred, average='micro', zero_division=0)
    r_wtd   = recall_score(y_test_num, y_pred, average='weighted', zero_division=0)
    f1_macro = f1_score(y_test_num, y_pred, average='macro', zero_division=0)
    f1_micro = f1_score(y_test_num, y_pred, average='micro', zero_division=0)
    f1_wtd   = f1_score(y_test_num, y_pred, average='weighted', zero_division=0)
    r2 = r2_score(y_test_num, y_pred)
    print(f"{name:12} | Acc: {acc:.4f} | P_mac: {p_macro:.4f} P_mic: {p_micro:.4f} P_wtd: {p_wtd:.4f} | R_mac: {r_macro:.4f} R_mic: {r_micro:.4f} R_wtd: {r_wtd:.4f} | F1_mac: {f1_macro:.4f} F1_mic: {f1_micro:.4f} F1_wtd: {f1_wtd:.4f} | R2: {r2:.4f}")

y_pred_lgbm = m_lgbm.predict(x_test_enc)
y_pred_xgb  = m_xgb.predict(x_test_enc)
y_pred_cat  = m_cat.predict(x_test_enc)

eval_model("LGBM",      y_pred_lgbm)
eval_model("XGBoost",   y_pred_xgb)
eval_model("CatBoost",  y_pred_cat)
eval_model("Stacking",  y_pred_ensemble)

# Матрица ошибок и отчёт по классам для стеккинга
print("\n--- Confusion matrix (Stacking) ---")
print("Классы:", list(le.classes_))
print(confusion_matrix(y_test_num, y_pred_ensemble))
print("\n--- Classification report (Stacking) ---")
print(classification_report(y_test_num, y_pred_ensemble, target_names=le.classes_, zero_division=0))

LGBM         | Acc: 0.5255 | P_mac: 0.5810 P_mic: 0.5255 P_wtd: 0.6155 | R_mac: 0.5443 R_mic: 0.5255 R_wtd: 0.5255 | F1_mac: 0.5138 F1_mic: 0.5255 F1_wtd: 0.5138 | R2: -0.7064
XGBoost      | Acc: 0.5398 | P_mac: 0.5988 P_mic: 0.5398 P_wtd: 0.6338 | R_mac: 0.5610 R_mic: 0.5398 R_wtd: 0.5398 | F1_mac: 0.5292 F1_mic: 0.5398 F1_wtd: 0.5278 | R2: -0.6354
CatBoost     | Acc: 0.5444 | P_mac: 0.5972 P_mic: 0.5444 P_wtd: 0.6309 | R_mac: 0.5613 R_mic: 0.5444 R_wtd: 0.5444 | F1_mac: 0.5373 F1_mic: 0.5444 F1_wtd: 0.5390 | R2: -0.6712
Stacking     | Acc: 0.5409 | P_mac: 0.5880 P_mic: 0.5409 P_wtd: 0.6200 | R_mac: 0.5581 R_mic: 0.5409 R_wtd: 0.5409 | F1_mac: 0.5304 F1_mic: 0.5409 F1_wtd: 0.5312 | R2: -0.6381

--- Confusion matrix (Stacking) ---
Классы: ['control', 'endo', 'exo']
[[3374  460 2991]
 [3346 3860 3294]
 [1155  324 6396]]

--- Classification report (Stacking) ---
              precision    recall  f1-score   support

     control       0.43      0.49      0.46      6825
        endo      

In [95]:
# Feature importance по базовым моделям и веса мета-модели
feat_names = x_train_enc.columns.tolist()

imp_lgbm = m_lgbm.feature_importances_
imp_xgb  = m_xgb.feature_importances_
imp_cat  = m_cat.get_feature_importance()

fi = pd.DataFrame({
    "feature": feat_names,
    "LGBM":     imp_lgbm,
    "XGBoost":  imp_xgb,
    "CatBoost": imp_cat,
})
# Нормализация 0–1 по каждой модели (масштабы у LGBM/XGB/Cat разные), затем средняя важность
fi_norm = fi[["LGBM", "XGBoost", "CatBoost"]].apply(
    lambda c: (c - c.min()) / (c.max() - c.min()) if c.max() > c.min() else c
)
fi["mean_imp"] = fi_norm.mean(axis=1)
fi = fi.sort_values("mean_imp", ascending=False).reset_index(drop=True)

print("--- Top-25 признаков (по средней важности) ---")
display(fi.head(25))

# Важность входов мета-модели (9 признаков: proba LGBM + XGB + CatBoost по 3 класса)
meta_feat = [f"LGBM_{c}" for c in le.classes_] + [f"XGB_{c}" for c in le.classes_] + [f"Cat_{c}" for c in le.classes_]
# coef_.shape = (n_classes, n_features)
meta_imp = np.abs(meta_learner.coef_).sum(axis=0)
meta_fi = pd.DataFrame({"meta_feature": meta_feat, "importance": meta_imp}).sort_values("importance", ascending=False)
print("\n--- Важность входов мета-модели (стеккинг) ---")
display(meta_fi)

--- Top-25 признаков (по средней важности) ---


,feature,LGBM,XGBoost,CatBoost,mean_imp
0,region_cortex,665,0.004008,8.187487,0.826631
1,region_cerebellum_right,430,0.004726,8.029503,0.732930
2,region_cerebellum_left,391,0.004375,6.849818,0.650426
3,region_cortex_left,473,0.003858,4.936198,0.591624
4,wave_1440.64,233,0.008086,2.854095,0.566323
5,wave_1439.57,206,0.006420,2.019697,0.447968
6,region_cortex_right,345,0.002888,2.896758,0.403181
7,wave_1438.50,129,0.004750,2.784097,0.369484
8,wave_1293.42,217,0.003891,2.307170,0.357652
9,region_striatum_left,355,0.002695,1.717655,0.352006



--- Важность входов мета-модели (стеккинг) ---


,meta_feature,importance
7,Cat_endo,2.235842
5,XGB_exo,2.067118
8,Cat_exo,1.787015
6,Cat_control,1.642550
1,LGBM_endo,1.380226
3,XGB_control,1.338948
4,XGB_endo,1.120708
0,LGBM_control,0.703418
2,LGBM_exo,0.520036


## Экспорт модели для веб-сервиса

Сохраняем модели в **нативных форматах** (без pickle): быстрая загрузка и совместимость версий XGBoost/LGBM/CatBoost. В папке `model_export/` появятся файлы моделей и `meta.joblib` с мета-моделью, кодировщиком и списком признаков.

In [96]:
import joblib
from pathlib import Path

export_dir = Path("model_export")
export_dir.mkdir(exist_ok=True)

# Нативные форматы — быстрая загрузка, без предупреждений XGBoost
m_xgb.get_booster().save_model(str(export_dir / "xgb.json"))
m_lgbm.booster_.save_model(str(export_dir / "lgbm.txt"))
m_cat.save_model(str(export_dir / "catboost.cbm"))

meta = {
    "meta_learner": meta_learner,
    "label_encoder": le,
    "feature_columns": x_train_enc.columns.tolist(),
    "wave_cols": wave_cols,
    "savgol_window": 11,
    "savgol_poly": 3,
}
joblib.dump(meta, export_dir / "meta.joblib")
print("Модели сохранены в", export_dir.absolute())
print("Файлы:", list(export_dir.iterdir()))

Модели сохранены в C:\Users\alexk\PycharmProjects\RomanSpectre\model_export
Файлы: [WindowsPath('model_export/catboost.cbm'), WindowsPath('model_export/lgbm.txt'), WindowsPath('model_export/meta.joblib'), WindowsPath('model_export/xgb.json')]
